# M2.0c Primer: Conductors, Images, Capacitance and Energy

Book pointer: chapter 7, sections 7.1 (conductors), 7.2 (ideal conductor in equilibrium), 7.3 (Laplace's equation with conductors), 7.4 (imaging methods and mirror charge); chapter 8, sections 8.1 (capacitance), 8.2 (combining capacitors), 8.3 (energy stored in an electric field).

Third of three primers for module 2. Primer 00a gave Gauss' law and primer 00b turned Poisson's equation into a linear system. This one supplies the physics that notebooks M2.3 and M2.4 use without stopping to explain it: what a conductor does to a field, how a conductor enters a solver, how the method of images produces exact solutions, what capacitance is, and why the energy of a charged system can be read off from the field alone. Every result is derived by hand with the numbers shown, then reproduced in code with the solver from primer 00b.

## 1. Warm-up

Try on paper first. Answers are in section 7.

1. Why must the electric field be zero inside a conductor in equilibrium, and what does that imply for the charge density inside it?
2. A neutral metal ball is placed in a uniform field. Describe what happens to its charge, and what the field looks like afterwards inside the ball and just outside its surface.
3. In the method of images, which two conditions must the image solution satisfy, and which theorem then guarantees that it is the solution?
4. Define capacitance. Does the capacitance of a pair of conductors change when you double the charge on them?
5. Derive the capacitance of a parallel-plate capacitor in three lines.
6. Write the energy of a charged capacitor in three equivalent forms, and the energy density of an electric field.
7. Two capacitors in series: what do they share, and what adds? Same question for parallel.

## 2. The ideas

A conductor is a body in which some charge is free to move. In a metal the mobile charges are electrons, in salt water they are ions, but the electrostatics is the same: if there is any electric field inside the material, the mobile charge feels a force and moves. Electrostatics is the study of situations in which nothing is moving any more, so the first fact about a conductor in equilibrium is that the field inside it is zero. Everything else about conductors follows from that one fact by applying the tools of the previous primers. Gauss' law applied to a box inside the metal says there is no net charge inside; any net charge must sit on the surface. The line integral of a zero field is zero, so every point of the conductor is at the same potential: the conductor is an equipotential. Just outside the surface the field must be perpendicular to it, because a component along the surface would push the surface charge sideways, and its magnitude is fixed by the surface charge through the pillbox argument of primer 00a, $E_n = \sigma/\epsilon_0$.

The second idea is induced charge. Put a neutral conductor into an external field and its mobile charges shift until the field inside is cancelled. Negative charge piles up on the side the field comes from, positive on the other side, and the total is still zero. This rearrangement is what a conductor does to every field it meets: it distorts the field outside, kills it inside, and converts a neutral body into a dipole. If the conductor is hollow, the same cancellation shields the cavity: no static external field reaches the inside of a closed metal box, which is why sensitive electronics live in metal enclosures.

The third idea is that a conductor is a boundary condition. Between conductors, where there is no charge, the potential satisfies Laplace's equation. On each conductor the potential is a known constant. That is exactly the Dirichlet boundary value problem of primer 00b, so a conductor enters a numerical solver as a block of grid nodes all fixed at one value. Once the solver has produced $V$, the field just outside each conductor gives the surface charge it carries, and Gauss' law on a box around the conductor gives its total charge. Nothing about the conductor needs to be modelled beyond "these nodes hold this value".

The fourth idea exploits uniqueness. Primer 00b showed that a solution of Laplace's equation with given boundary values is the only one. So if you can guess a potential that satisfies Laplace's equation in the region of interest and takes the right values on every conductor, your guess is the answer, however you arrived at it. The method of images is a systematic way of guessing: replace the conductor by one or more fictitious charges, placed outside the region of interest, chosen so that the potential of the real charges plus the fictitious ones is constant on the conductor's surface. The classic cases are a point charge in front of a grounded plane, whose image is the mirror charge, and a point charge outside a grounded sphere, whose image is a smaller charge inside the sphere. Both give closed forms for everything: the potential, the field, the induced surface charge, and the force on the real charge. Notebook M2.3 uses the plane image as the exact answer that its relaxation solver has to reproduce.

The fifth idea is capacitance. Take two conductors, put charge $+Q$ on one and $-Q$ on the other, and measure the potential difference $V$ between them. Because Laplace's equation is linear, doubling $Q$ doubles every field and doubles $V$. The ratio $C = Q/V$ is therefore independent of $Q$: it is a number fixed by the shapes and positions of the two conductors, called their capacitance, measured in farads (coulombs per volt). Large capacitance means a lot of charge is stored per volt, which happens when the conductors are large and close together. For a few geometries with enough symmetry, Gauss' law gives $E$, a line integral gives $V$, and $C$ follows in three lines. For every other geometry, a numerical solver does the same job.

The last idea is energy. Charging a capacitor takes work, because each new piece of charge is pushed against the potential difference already present, and the total work comes to $\tfrac12 CV^2$. That energy can be assigned to the field itself, with an energy density $\tfrac12\epsilon_0E^2$ at every point in space. The two descriptions agree exactly, and the second is more useful numerically: integrate $\tfrac12\epsilon_0 E^2$ over the grid, set it equal to $\tfrac12 CV^2$, and you have the capacitance without ever counting charge on a staircased conductor surface. That is the whole method of notebook M2.4, and it works for any shape the grid can hold.

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from emlib import fields, relax
from emlib.fields import EPS0, K

plt.rcParams["figure.figsize"] = (6, 5)
np.set_printoptions(precision=4, suppress=True)

### 3.1 The ideal conductor in equilibrium

An ideal conductor has an unlimited supply of mobile charge and no resistance to its motion. In equilibrium it has five properties, and each is a one-line consequence of the one before it.

1. $\mathbf{E} = 0$ inside. If it were not, mobile charge would move, and the situation would not be equilibrium.
2. $\rho = 0$ inside. Gauss' law in differential form: $\nabla\cdot\mathbf{E} = \rho/\epsilon_0$, and the left side is zero.
3. All net charge sits on the surface. It cannot be inside (property 2), so the only place left is the surface, where it forms a surface density $\sigma$.
4. The conductor is an equipotential. $V_b - V_a = -\int_a^b\mathbf{E}\cdot d\mathbf{l}$ along any path inside the metal, and the integrand is zero.
5. Just outside, $\mathbf{E}$ is normal to the surface with $E_n = \sigma/\epsilon_0$. A tangential component would move the surface charge along the surface, violating equilibrium. The magnitude comes from the pillbox of primer 00a, section 3.7: one face inside the metal sees no field, the other sees $E_n$, so $E_nA = \sigma A/\epsilon_0$.

Property 4 is the one a solver uses, and property 5 is how the solver's answer is turned back into charge.

Induced charge. Place a neutral conductor in an external field $\mathbf{E}_0$. The mobile charge moves until the field it produces cancels $\mathbf{E}_0$ everywhere inside. The result is a surface charge that is negative where the field lines enter and positive where they leave, with zero total. From outside, the conductor now looks like a dipole added to the original field. The field lines outside bend to meet the surface at right angles, and the field just outside is strongest where the surface curves most sharply, which is why charge concentrates at points and edges.

Shielding. If the conductor has a cavity with no charge inside it, the field in the cavity is zero whatever happens outside. The argument is uniqueness again: the cavity wall is an equipotential, Laplace's equation holds inside, and the constant potential satisfies both, so it is the solution and its gradient vanishes. This is the Faraday cage. The converse does not hold: a charge inside the cavity does produce a field outside, through the induced charges on the inner and outer surfaces (primer 00a, worked example 14).

**Worked example 1 (conducting sphere in a uniform field).** A neutral conducting sphere of radius $R$ sits in a uniform field $E_0\hat{\mathbf{z}}$. Solving Laplace's equation outside the sphere with $V$ constant on the surface gives the surface charge
$$\sigma(\theta) = 3\epsilon_0E_0\cos\theta,$$
with $\theta$ the angle from the field direction. Take that as given and check what it implies. Total charge: $\int\sigma\,dA = 3\epsilon_0E_0R^2\int_0^{2\pi}\!d\phi\int_0^\pi\cos\theta\sin\theta\,d\theta = 3\epsilon_0E_0R^2\cdot2\pi\cdot0 = 0$. Neutral, as it must be. Dipole moment, $p_z = \int\sigma\,z\,dA$ with $z = R\cos\theta$: $3\epsilon_0E_0R^3\cdot2\pi\int_0^\pi\cos^2\theta\sin\theta\,d\theta = 3\epsilon_0E_0R^3\cdot2\pi\cdot\tfrac23 = 4\pi\epsilon_0R^3E_0$. So a metal ball of radius $R$ in a field $E_0$ is a dipole of moment $4\pi\epsilon_0R^3E_0$, proportional to its volume. The field just outside the pole ($\theta = 0$) is $\sigma/\epsilon_0 = 3E_0$: three times the applied field. For $R = 1\,$cm and $E_0 = 10^4\,$V/m, $p = 4\pi\cdot8.854\times10^{-12}\cdot10^{-6}\cdot10^4 = 1.11\times10^{-12}\,$C m.

**Worked example 2 (charge sharing).** Two conducting spheres of radii $R_1 = 1\,$cm and $R_2 = 3\,$cm, far apart, are connected by a thin wire and given a total charge $Q = 4\,$nC. Connected conductors form one equipotential, so $q_1/4\pi\epsilon_0R_1 = q_2/4\pi\epsilon_0R_2$ (each sphere's potential is that of a point charge at its centre, since they are far apart). Hence $q_1/q_2 = R_1/R_2 = 1/3$: $q_1 = 1\,$nC, $q_2 = 3\,$nC. The surface densities are $\sigma_1 = q_1/4\pi R_1^2 = 7.96\times10^{-7}\,$C/m$^2$ and $\sigma_2 = 2.65\times10^{-7}\,$C/m$^2$: the smaller sphere holds less charge but a higher density, $\sigma\propto1/R$. The field at its surface, $\sigma/\epsilon_0$, is three times larger. Sharp points on a conductor are the limit $R\to0$ of this, which is why lightning rods and corona discharge start at points.

The code integrates $\sigma(\theta)$ over the sphere numerically for the total charge and dipole moment, and reproduces the charge-sharing numbers.

**Animation.** A neutral conducting slab, drawn as fixed positive ions with one mobile electron beside each, is placed in an applied field $E_0$. The electrons drift against the field. Every ion inside keeps an electron beside it, so the interior stays neutral, but the left face gains a layer of extra electrons and the right face is left with uncovered ions. The field of that separated surface charge opposes $E_0$ inside the slab, and the readout shows the interior field falling to zero, at which point the electrons stop. For a slab wide compared with its thickness, the two face layers carry equal and opposite charge and their fields cancel outside, so the applied field there is unchanged. The shift is drawn hugely exaggerated; the caption gives the real one for copper.

In [ ]:
from emlib import show
show.animation("m2_electrostatics", "ConductorEquilibrium")

In [ ]:
R, E0 = 0.01, 1e4
theta = np.linspace(0, np.pi, 2001)
sigma = 3 * EPS0 * E0 * np.cos(theta)
dA = 2 * np.pi * R**2 * np.sin(theta)                       # area of the ring between theta and theta + dtheta, per dtheta
Q_total = np.trapezoid(sigma * dA, theta)
p_z = np.trapezoid(sigma * R * np.cos(theta) * dA, theta)
print(f"sphere in a uniform field: total induced charge = {Q_total:.2e} C   dipole moment = {p_z:.4e} C m   4 pi eps0 R^3 E0 = {4 * np.pi * EPS0 * R**3 * E0:.4e} C m")
print(f"field just outside the pole: sigma/eps0 = {sigma[0] / EPS0:.0f} V/m = 3 E0")

# Worked example 2: charge sharing between connected spheres.
R1, R2, Q = 0.01, 0.03, 4e-9
q1 = Q * R1 / (R1 + R2); q2 = Q - q1
print(f"connected spheres: q1 = {q1 * 1e9:.2f} nC, q2 = {q2 * 1e9:.2f} nC;  potentials {K * q1 / R1:.1f} V and {K * q2 / R2:.1f} V (equal)")
print(f"surface densities: {q1 / (4 * np.pi * R1**2):.3e} and {q2 / (4 * np.pi * R2**2):.3e} C/m^2, ratio {R2 / R1:.1f}")

### 3.2 Conductors in a Laplace solver

Between conductors $V$ obeys Laplace's equation, and on each conductor $V$ is a known constant. So in the language of primer 00b a conductor is a set of nodes marked `fixed = True` with `V0` holding its potential. The solver never updates them, and the free nodes around them adjust. That is the entire treatment.

The solution is then read back into charge in two ways.

Surface charge. Property 5 says $\sigma = \epsilon_0E_n$ with $\hat{\mathbf{n}}$ the outward normal of the conductor. On a grid, the field at the surface comes from the one-sided difference between the surface node and the first free node outside it: for a face whose outward normal is $+\hat{\mathbf{x}}$, $E_x \approx -(V_{i+1,j} - V_{i,j})/h$ with $i$ the surface node. Notebook M2.3 does exactly this on a grounded plane.

Total charge. Gauss' law: the charge on a conductor is $\epsilon_0$ times the outward flux of $\mathbf{E}$ through any closed surface that encloses it and nothing else. On the grid that surface is a box of nodes just outside the conductor, and `fields.box_flux` sums the flux through its faces. The box route is more accurate than summing $\sigma$ over the surface, because it uses central differences on free nodes instead of one-sided differences at the surface.

**Worked example 3.** A square conductor of side $s$ is held at the midplane potential between two plates at $0$ and $V_0$ a distance $L$ apart, with the conductor centred between them. By symmetry (reflect through the midplane and swap $V\to V_0 - V$, which flips the sign of every charge) the induced charges on the two faces are equal and opposite, so the total is zero, while each face carries about $\pm\epsilon_0E_0s$ with $E_0 = V_0/L$: with $V_0 = 1\,$V, $L = 1\,$m, $s = 0.2\,$m, that is $\pm1.77\times10^{-12}\,$C per metre of depth. The field at the faces normal to $E_0$ is larger than $E_0$ because the conductor shortens the gap the potential has to drop across. If instead the conductor is grounded ($V = 0$) and a positive line charge is placed near it, the induced charge is negative: field lines from the line charge end on the conductor.

The code solves both cases on an $81\times81$ grid with `relax.solve_sparse`.

In [ ]:
n = 81
x = np.linspace(0, 1, n); h = x[1] - x[0]
X, Y = np.meshgrid(x, x, indexing="ij")
block = (np.abs(X - 0.5) <= 0.1 + 1e-9) & (np.abs(Y - 0.5) <= 0.1 + 1e-9)       # conductor: 0.4 <= x, y <= 0.6
i_lo, i_hi = np.argmin(np.abs(x - 0.375)), np.argmin(np.abs(x - 0.625))        # a Gauss box two cells outside the block

# Case 1: plates at x = 0 (0 V) and x = 1 (1 V), sides fixed to the linear ramp, block held at the midplane value 0.5 V.
V0 = np.zeros((n, n)); fixed = np.zeros((n, n), dtype=bool)
V0[-1, :] = 1.0; V0[:, 0] = V0[:, -1] = x
V0[block] = 0.5; fixed[block] = True
V = relax.solve_sparse(V0, fixed)
E = -fields.gradient(V, h, h)
Q_box = EPS0 * fields.box_flux(E, (h, h), (i_lo, i_lo), (i_hi, i_hi))
i_face = np.nonzero(block[:, 40])[0]                                            # first and last block row
sigma_hi = EPS0 * (-(V[i_face[-1] + 1, :] - V[i_face[-1], :]) / h)             # outward normal +x: E_x by one-sided difference
sigma_lo = EPS0 * (-(V[i_face[0] - 1, :] - V[i_face[0], :]) / h)               # outward normal -x: E_n = -E_x = (V_i - V_{i-1}) / h
face = block[40, :]
print(f"conductor at the midplane potential in a uniform field {1.0:.0f} V/m:")
print(f"  charge on the face towards the 1 V plate: {np.trapezoid(sigma_hi[face], dx=h):+.3e} C/m   towards the 0 V plate: {np.trapezoid(sigma_lo[face], dx=h):+.3e} C/m")
print(f"  total from the Gauss box: {Q_box:+.2e} C/m   (eps0 E0 s = {EPS0 * 0.2:.2e} C/m per face; the total vanishes by symmetry)")

# Case 2: grounded block, grounded outer boundary, a positive line charge lambda at (0.8, 0.5).
lam = 1e-9
rho = np.zeros((n, n)); rho[np.argmin(np.abs(x - 0.8)), np.argmin(np.abs(x - 0.5))] = lam / h**2
V0b = np.zeros((n, n)); fixed_b = block.copy()
Vb = relax.solve_sparse(V0b, fixed_b, rho=rho, h=h)
Eb = -fields.gradient(Vb, h, h)
Q_induced = EPS0 * fields.box_flux(Eb, (h, h), (i_lo, i_lo), (i_hi, i_hi))
print(f"grounded conductor next to a line charge {lam:.0e} C/m: induced charge = {Q_induced:+.3e} C/m = {Q_induced / lam:+.3f} lambda  (the rest of the field lines end on the grounded box)")

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
for ax, VV, title in zip(axes, (V, Vb), ("conductor at 0.5 V between plates", "grounded conductor and a line charge")):
    cs = ax.contour(X, Y, VV, levels=21, cmap="viridis"); ax.clabel(cs, fontsize=6)
    ax.add_patch(plt.Rectangle((0.4, 0.4), 0.2, 0.2, color="grey")); ax.set_aspect("equal"); ax.set_title(title)
plt.show()

### 3.3 The method of images: a plane

Uniqueness (primer 00b, section 3.7) says that any potential which satisfies Laplace's equation in a region and takes the prescribed values on its boundary is the solution there. The method of images uses that freedom. Remove the conductor, keep the region of interest, and put fictitious charges in the space the conductor used to occupy. If they make the potential constant on the surface where the conductor was, the potential in the region of interest is exact. The fictitious charges live outside the region, so they do not add any sources there, and Laplace's equation still holds.

**Worked example 4 (point charge and a grounded plane).** A charge $q$ sits at height $d$ above an infinite grounded conducting plane $z = 0$. Guess: replace the plane by a charge $-q$ at $z = -d$, the mirror image. The potential of the pair at a point $(x, y, z)$ is
$$V = \frac{q}{4\pi\epsilon_0}\left(\frac{1}{\sqrt{x^2 + y^2 + (z - d)^2}} - \frac{1}{\sqrt{x^2 + y^2 + (z + d)^2}}\right).$$
On the plane $z = 0$ the two square roots are equal and $V = 0$: the boundary condition is met. Far away $V\to0$. Above the plane there is only the real charge, so Laplace's equation holds everywhere except at $q$. By uniqueness this is the potential above the plane. Below the plane the real answer is $V = 0$ (inside the conductor), not the image formula; the image is a device for the region above the plane only.

Induced surface charge. On the plane the field is normal, $E_z = -\partial V/\partial z$ at $z = 0$. Differentiating the two terms gives $E_z(s) = -\dfrac{2qd}{4\pi\epsilon_0(s^2 + d^2)^{3/2}}$ with $s = \sqrt{x^2 + y^2}$ the distance from the foot of the perpendicular, so
$$\sigma(s) = \epsilon_0E_z = -\frac{qd}{2\pi(s^2 + d^2)^{3/2}}.$$
It is largest directly under the charge, $\sigma(0) = -q/2\pi d^2$, and falls like $1/s^3$. Total induced charge: $\int_0^\infty\sigma\,2\pi s\,ds = -qd\int_0^\infty\dfrac{s\,ds}{(s^2 + d^2)^{3/2}} = -qd\left[-\dfrac{1}{\sqrt{s^2 + d^2}}\right]_0^\infty = -qd\cdot\dfrac1d = -q$. Every field line from $q$ ends on the plane.

Force on the charge. Above the plane the field is exactly that of the image, so $q$ is attracted towards the plane with $F = \dfrac{q^2}{4\pi\epsilon_0(2d)^2}$. For $q = 1\,$nC at $d = 1\,$cm: $F = 8.988\times10^9\cdot10^{-18}/(4\times10^{-4}) = 2.25\times10^{-5}\,$N.

**Worked example 5 (the two-dimensional version).** Notebook M2.3 solves on a 2D grid, which represents systems uniform along $z$, so its point source is an infinite line charge $\lambda$. Its field, from primer 00a worked example 8, is $E_r = \lambda/2\pi\epsilon_0r$, and its potential follows by integrating inward from a reference radius $r_0$: $V(r) = -\int_{r_0}^r E_r\,dr = -\dfrac{\lambda}{2\pi\epsilon_0}\ln\dfrac{r}{r_0}$. The potential of a line charge diverges at infinity, so $V = 0$ is set at a finite $r_0$ instead; only differences matter. A line charge $\lambda$ at distance $d$ from a grounded plane has the image $-\lambda$ at the mirror position, and the pair's potential is
$$V = \frac{\lambda}{2\pi\epsilon_0}\ln\frac{r_-}{r_+},$$
with $r_\pm$ the distances to the real and image lines. The reference radii cancel in the difference. On the plane $r_- = r_+$ and $V = 0$. The induced charge is $\sigma(y) = \epsilon_0E_x = -\dfrac{\lambda d}{\pi(d^2 + y^2)}$, and its integral is $-\dfrac{\lambda d}{\pi}\cdot\dfrac{\pi}{d} = -\lambda$ per unit depth. These are the two formulas notebook M2.3 fixes on its outer boundary and checks on its plane.

The code evaluates the two image potentials on their planes and integrates both induced charge densities.

**Animation.** A charge $+q$ moves from $d = 3$ cm to $d = 0.5$ cm above a grounded plane. Its image $-q$ (dashed) mirrors it below, and the plot on the right is the induced surface charge $\sigma(s)$ of worked example 4. As the charge approaches, the induced charge piles up underneath it: $\sigma(0) = -q/2\pi d^2$ grows and the profile narrows, while its total stays $-q$. The yellow arrows are the field at the plane, normal to it and proportional to $|\sigma|$. The readout gives the attraction $q^2/4\pi\epsilon_0(2d)^2$ for $q = 1$ nC, $2.25\times10^{-5}$ N at 1 cm.

In [ ]:
from emlib import show
show.animation("m2_electrostatics", "ImageCharge")

In [ ]:
q, d = 1e-9, 0.01
V_plane = lambda pts: fields.point_charges_V([[0, 0, d], [0, 0, -d]], [q, -q], pts)
pts = np.array([[0, 0, 0], [0.02, 0, 0], [0.01, 0.03, 0], [1.0, -2.0, 0]])
print("point charge above a grounded plane: V on the plane =", V_plane(pts), " (zero)")
s = np.linspace(0, 2.0, 200001)
sigma_s = -q * d / (2 * np.pi * (s**2 + d**2) ** 1.5)
print(f"induced charge on the plane: {np.trapezoid(sigma_s * 2 * np.pi * s, s):.4e} C   (-q = {-q:.1e} C; the tail beyond 2 m holds the rest)")
print(f"sigma directly under the charge: {sigma_s[0]:.3e} C/m^2 = -q / (2 pi d^2) = {-q / (2 * np.pi * d**2):.3e}")
print(f"force on the charge: {K * q**2 / (2 * d) ** 2:.3e} N")

# The 2D line-charge image: V = (lambda / 2 pi eps0) ln(r_minus / r_plus), zero on the plane x = 0.
lam, d2 = 1e-9, 0.3
def V_line_image(x, y):
    r_plus = np.hypot(x - d2, y); r_minus = np.hypot(x + d2, y)
    return lam / (2 * np.pi * EPS0) * np.log(r_minus / r_plus)
yy = np.linspace(-20, 20, 400001)
print("line charge and grounded plane: V at x = 0 =", V_line_image(np.zeros(4), np.array([0.0, 0.1, 0.5, 3.0])), " (zero)")
sigma_y = -lam * d2 / (np.pi * (d2**2 + yy**2))
print(f"induced charge per unit depth: {np.trapezoid(sigma_y, yy):.4e} C/m   (-lambda = {-lam:.1e})")

**Worked example 6 (point charge and a grounded sphere).** A charge $q$ sits at distance $d$ from the centre of a grounded conducting sphere of radius $R < d$. The image is a charge
$$q' = -\frac{R}{d}\,q \quad\text{at distance}\quad d' = \frac{R^2}{d}\ \text{from the centre, on the line to } q.$$
Check at the two points of the sphere on that line. Nearest point, distance $d - R$ from $q$ and $R - d'$ from $q'$:
$$V \propto \frac{q}{d - R} + \frac{q'}{R - R^2/d} = \frac{q}{d - R} - \frac{Rq/d}{R(d - R)/d} = \frac{q}{d - R} - \frac{q}{d - R} = 0.$$
Farthest point, distance $d + R$ from $q$ and $R + d'$ from $q'$:
$$V \propto \frac{q}{d + R} - \frac{Rq/d}{R(d + R)/d} = 0.$$
It vanishes at every other point of the sphere as well (the code checks four hundred of them), so by uniqueness the potential outside the sphere is that of $q$ and $q'$. The total induced charge on the sphere is $q' = -qR/d$, less than $q$ in magnitude: some field lines from $q$ escape to infinity. With $R = 1$, $d = 3$: $q' = -q/3$ at $d' = 1/3$. If the sphere is isolated and neutral instead of grounded, add a third charge $-q'$ at the centre; it keeps the sphere an equipotential (at a nonzero value) and restores zero total charge.

In [ ]:
R_s, d_s, q = 1.0, 3.0, 1e-9
q_img, d_img = -q * R_s / d_s, R_s**2 / d_s
k = np.arange(400) + 0.5
th, ph = np.arccos(1 - 2 * k / 400), np.pi * (1 + 5**0.5) * k
surface = R_s * np.stack([np.sin(th) * np.cos(ph), np.sin(th) * np.sin(ph), np.cos(th)], axis=-1)
V_surf = fields.point_charges_V([[0, 0, d_s], [0, 0, d_img]], [q, q_img], surface)
print(f"image charge {q_img / q:+.4f} q at distance {d_img:.4f} from the centre")
print(f"|V| on 400 points of the sphere: max {np.abs(V_surf).max():.2e} V, compared with K q / d = {K * q / d_s:.2e} V")

# Picture: equipotentials of q and its image, with the sphere drawn in.
xs = np.linspace(-2.5, 5.0, 301); zs = np.linspace(-3.0, 3.0, 241)
Xs, Zs = np.meshgrid(xs, zs, indexing="ij")
P = np.stack([Xs, np.zeros_like(Xs), Zs], axis=-1)           # take the sphere's axis along x in this picture
Vxz = fields.point_charges_V([[d_s, 0, 0], [d_img, 0, 0]], [q, q_img], P)
outside = np.hypot(Xs, Zs) >= R_s
fig, ax = plt.subplots(figsize=(7.5, 6))
ax.contour(Xs, Zs, np.where(outside, Vxz, np.nan), levels=np.linspace(-1, 6, 29), cmap="coolwarm")
ax.add_patch(plt.Circle((0, 0), R_s, color="lightgrey")); ax.plot(d_s, 0, "ko", label="q"); ax.plot(d_img, 0, "wx", label="image q'")
ax.set_aspect("equal"); ax.legend(); ax.set_title("point charge outside a grounded sphere: V = 0 on the sphere"); plt.show()

### 3.4 Capacitance

Two conductors carrying $+Q$ and $-Q$ have a potential difference $V$ between them. Because the equations are linear, $V$ is proportional to $Q$, and the constant of proportionality is set by geometry alone:
$$Q = CV, \qquad C = \frac{Q}{V}\ \text{(farad, F = C/V)}.$$
The farad is enormous. Everyday capacitors are picofarads to microfarads; a one-farad capacitor the size of a coffee cup is a recent achievement. An isolated single conductor also has a capacitance, defined with the second conductor at infinity: $C = Q/V$ with $V$ the potential relative to infinity.

Each symmetric geometry is done the same way: put $\pm Q$ on the conductors, get $E$ from Gauss' law (primer 00a), integrate $E$ from one conductor to the other to get $V$, divide.

**Worked example 7 (parallel plates).** Two plates of area $A$ a distance $d$ apart, $d$ small compared with the plate size so that the field between them is uniform and the fringes can be ignored. Surface charge $\sigma = Q/A$. From primer 00a, worked example 12, the field between oppositely charged sheets is $E = \sigma/\epsilon_0 = Q/\epsilon_0A$ and zero outside. $V = Ed = Qd/\epsilon_0A$. So
$$C = \frac{\epsilon_0A}{d}.$$
With $A = 1\,$cm$^2$ and $d = 1\,$mm: $C = 8.854\times10^{-12}\cdot10^{-4}/10^{-3} = 0.885\,$pF. Per unit depth in a 2D model, with plates of width $W$: $C = \epsilon_0W/d$ farads per metre.

**Worked example 8 (coaxial cylinders).** Inner radius $a$, outer radius $b$, charge $\lambda$ per unit length on the inner conductor and $-\lambda$ on the outer. Between them, by the cylindrical Gauss argument, $E = \lambda/2\pi\epsilon_0r$. Then $V = \int_a^b\dfrac{\lambda\,dr}{2\pi\epsilon_0r} = \dfrac{\lambda}{2\pi\epsilon_0}\ln\dfrac ba$, so per unit length
$$C = \frac{2\pi\epsilon_0}{\ln(b/a)}.$$
A coaxial cable with $a = 1\,$mm and $b = 3.5\,$mm: $C = 5.563\times10^{-11}/\ln3.5 = 44.4\,$pF/m (times the dielectric constant of the insulation, which module 3 adds).

**Worked example 9 (concentric spheres and an isolated sphere).** Inner radius $a$, outer $b$, charges $\pm Q$. Between them $E = Q/4\pi\epsilon_0r^2$ and $V = \dfrac{Q}{4\pi\epsilon_0}\left(\dfrac1a - \dfrac1b\right) = \dfrac{Q}{4\pi\epsilon_0}\dfrac{b - a}{ab}$, so
$$C = 4\pi\epsilon_0\frac{ab}{b - a}.$$
Let $b\to\infty$: $C = 4\pi\epsilon_0a$, the capacitance of an isolated sphere. A sphere of radius 1 m has $C = 111\,$pF. The Earth, $R = 6.37\times10^6\,$m, has $C = 709\,\mu$F. Check the parallel-plate limit too: with $b = a + d$ and $d\ll a$, $ab/(b - a)\approx a^2/d$ and $C\approx4\pi a^2\epsilon_0/d = \epsilon_0A/d$.

The code computes these numbers, then does the coaxial case with no formula at all: the solver from primer 00b on a square grid with the inner disk fixed at $1\,$V and everything at radius $b$ fixed at $0$, the charge read off by Gauss' law on a box between the conductors, and $C = Q/V$. The circles are staircased onto the grid, so the error is set by how well the grid resolves the inner radius, and it shrinks as the grid is refined.

In [ ]:
print(f"parallel plates, A = 1 cm^2, d = 1 mm:        C = {EPS0 * 1e-4 / 1e-3 * 1e12:.3f} pF")
print(f"coaxial cable, a = 1 mm, b = 3.5 mm:           C = {2 * np.pi * EPS0 / np.log(3.5) * 1e12:.1f} pF/m")
print(f"concentric spheres, a = 10 cm, b = 20 cm:      C = {4 * np.pi * EPS0 * 0.1 * 0.2 / 0.1 * 1e12:.2f} pF")
print(f"isolated sphere, R = 1 m:                      C = {4 * np.pi * EPS0 * 1.0 * 1e12:.1f} pF;   the Earth: {4 * np.pi * EPS0 * 6.37e6 * 1e6:.0f} uF")

def coax_solution(n, a=0.25, b=1.0, V0=1.0):
    xs = np.linspace(-b, b, n); hc = xs[1] - xs[0]
    Xc, Yc = np.meshgrid(xs, xs, indexing="ij"); Rc = np.hypot(Xc, Yc)
    Vc = np.zeros((n, n)); fx = np.zeros((n, n), dtype=bool)
    Vc[Rc <= a] = V0; fx[Rc <= a] = True                      # inner conductor at V0
    fx[Rc >= b] = True                                         # outer conductor at 0
    return relax.solve_sparse(Vc, fx), hc, xs

a, b = 0.25, 1.0
C_exact = 2 * np.pi * EPS0 / np.log(b / a)
print("\ncoaxial capacitor on a grid, a = 0.25, b = 1, inner conductor at 1 V:")
for m in (51, 101, 201):
    Vc, hc, xs = coax_solution(m, a, b)
    Ec = -fields.gradient(Vc, hc, hc)
    k0, k1 = np.argmin(np.abs(xs + 0.6)), np.argmin(np.abs(xs - 0.6))            # a square Gauss box at |x|,|y| = 0.6, between the conductors
    Q_grid = EPS0 * fields.box_flux(Ec, (hc, hc), (k0, k0), (k1, k1))
    print(f"  n = {m:3d}, h = {hc:.4f}:  C = Q/V = {Q_grid:.4e} F/m   closed form {C_exact:.4e} F/m   ratio {Q_grid / C_exact:.4f}")
Vc, hc, xs = coax_solution(101, a, b)
Xc, Yc = np.meshgrid(xs, xs, indexing="ij")
plt.contour(Xc, Yc, Vc, levels=np.linspace(0, 1, 21), cmap="viridis"); plt.gca().set_aspect("equal"); plt.colorbar(label="V")
plt.title("coaxial capacitor: V falls as ln(b/r)"); plt.show()

### 3.5 Combining capacitors

Two capacitors connected in a circuit act as one, and the rule for the combined capacitance follows from what the connection forces to be equal.

Parallel. Both capacitors have the same two terminals, so they share the same voltage $V$. The charges add: $Q = C_1V + C_2V$, hence
$$C_{\parallel} = C_1 + C_2.$$
Physically, parallel plates side by side are one bigger plate: $\epsilon_0(A_1 + A_2)/d$.

Series. The capacitors are in a chain, and the plate pair in the middle is an isolated conductor. It started neutral, so whatever charge $+Q$ ends up on one of its plates, $-Q$ is on the other: every capacitor in the chain carries the same $Q$. The voltages add: $V = Q/C_1 + Q/C_2$, hence
$$\frac{1}{C_{\text{series}}} = \frac{1}{C_1} + \frac{1}{C_2}.$$
Physically, two gaps in series are one longer gap: $d_1 + d_2$ in the denominator. The series combination is always smaller than the smaller of the two.

**Worked example 10.** $C_1 = 2\,\mu$F and $C_2 = 6\,\mu$F across $12\,$V. In parallel: $C = 8\,\mu$F, $Q_1 = 24\,\mu$C, $Q_2 = 72\,\mu$C. In series: $1/C = 1/2 + 1/6 = 2/3$, $C = 1.5\,\mu$F, $Q = 18\,\mu$C on each, $V_1 = 18/2 = 9\,$V, $V_2 = 18/6 = 3\,$V. The smaller capacitor takes the larger share of the voltage, and the two shares add to 12.

The floating plate of notebook M2.4 is the series case seen from the solver's side. Three plates at $V = 1$, unknown, and $0$ are two capacitors in series with the middle plate the shared isolated conductor. A Dirichlet solver needs a value for the middle plate, and the physical value is the one for which it carries zero net charge. Section 3.6 shows that this is also the value that minimises the field energy, which is how the notebook finds it.

In [ ]:
C1, C2, V = 2e-6, 6e-6, 12.0
C_par, C_ser = C1 + C2, 1 / (1 / C1 + 1 / C2)
Q_ser = C_ser * V
print(f"parallel: C = {C_par * 1e6:.1f} uF, charges {C1 * V * 1e6:.0f} uC and {C2 * V * 1e6:.0f} uC")
print(f"series:   C = {C_ser * 1e6:.1f} uF, charge {Q_ser * 1e6:.0f} uC on each, voltages {Q_ser / C1:.0f} V + {Q_ser / C2:.0f} V = {V:.0f} V")

### 3.6 Energy

Charging a capacitor is work. Suppose charge $q$ has already been moved from the negative plate to the positive one, so the voltage is $q/C$. Moving the next piece $dq$ across costs $dW = (q/C)\,dq$. Integrating from $0$ to $Q$:
$$U = \int_0^Q\frac{q}{C}\,dq = \frac{Q^2}{2C} = \frac12QV = \frac12CV^2.$$
Three forms of one number, related by $Q = CV$. The factor $\tfrac12$ is there because the voltage grew from zero while the charge was being moved; the average voltage during charging was $V/2$.

Energy of a set of charges. The same reasoning for point charges assembled one at a time gives $U = \tfrac12\sum_i q_iV_i$, where $V_i$ is the potential at charge $i$ due to all the others. The $\tfrac12$ stops each pair being counted twice. For three charges of $1\,$nC at the corners of an equilateral triangle of side $1\,$cm, the three pairs each contribute $kq^2/r = 8.988\times10^9\cdot10^{-18}/0.01 = 8.99\times10^{-7}\,$J, total $2.70\times10^{-6}\,$J.

Energy in the field. For the parallel-plate capacitor, $U = \tfrac12CV^2 = \tfrac12\dfrac{\epsilon_0A}{d}(Ed)^2 = \tfrac12\epsilon_0E^2\cdot Ad$. The factor $Ad$ is the volume between the plates, the only place where the field is nonzero. So the energy is $\tfrac12\epsilon_0E^2$ per unit volume of field. That turns out to hold in general, not just for uniform fields:
$$U = \int\frac12\epsilon_0E^2\,dv \quad\text{over all space}.$$
This is a different way of thinking about where the energy is. The charge picture puts it in the configuration of the charges; the field picture spreads it through space wherever there is field. For static problems the two always agree. The field picture is the one that survives into electrodynamics, where fields carry energy away from the charges altogether (chapter 15), and it is the one that is convenient on a grid.

**Worked example 11 (spherical capacitor, both routes).** $a = 10\,$cm, $b = 20\,$cm, $Q = 1\,$nC. From worked example 9, $C = 22.25\,$pF, so $U = Q^2/2C = 10^{-18}/(4.450\times10^{-11}) = 2.247\times10^{-8}\,$J. Field route: between the shells $E = Q/4\pi\epsilon_0r^2$ and the volume element is $4\pi r^2\,dr$, so
$$U = \int_a^b\frac12\epsilon_0\left(\frac{Q}{4\pi\epsilon_0r^2}\right)^2 4\pi r^2\,dr = \frac{Q^2}{8\pi\epsilon_0}\int_a^b\frac{dr}{r^2} = \frac{Q^2}{8\pi\epsilon_0}\left(\frac1a - \frac1b\right) = \frac{10^{-18}\cdot5}{2.225\times10^{-10}} = 2.247\times10^{-8}\,\text{J}.$$
Same number. Letting $b\to\infty$ gives the energy stored in the field of an isolated charged sphere, $Q^2/8\pi\epsilon_0a$; letting $a\to0$ makes it infinite, which is the well-known trouble with the self-energy of a point charge.

Capacitance from energy. Turn the last equality around: solve the boundary value problem with one conductor at $V_0$ and the other at $0$, integrate $\tfrac12\epsilon_0E^2$ over the grid to get $U$, and then
$$C = \frac{2U}{V_0^2}.$$
No charge is counted anywhere, so the staircasing of the conductor surface enters only through the field, which is smooth. This is the method of notebook M2.4, and `relax.field_energy` is the integral. The floating plate of section 3.5 fits here too: for fixed charges on the outer plates, the true potential of an isolated middle plate is the one that minimises the total field energy, so scanning its potential and picking the minimum gives both its potential and the series capacitance.

The code does the triangle of charges, the spherical capacitor both ways, and the parallel-plate capacitor on a grid by the energy route.

In [ ]:
# Energy of three equal charges at the corners of an equilateral triangle: (1/2) sum q_i V_i.
qs = np.full(3, 1e-9); side = 0.01
corners = side * np.array([[0, 0, 0], [1, 0, 0], [0.5, np.sqrt(3) / 2, 0]])
V_at = fields.point_charges_V(corners, qs, corners)                 # each charge's own term is skipped by the library
print(f"three 1 nC charges on a 1 cm triangle: U = (1/2) sum q V = {0.5 * np.sum(qs * V_at):.3e} J   (3 k q^2 / r = {3 * K * 1e-18 / side:.3e} J)")

# Spherical capacitor: charge route and field route.
a_s, b_s, Q = 0.1, 0.2, 1e-9
C_sph = 4 * np.pi * EPS0 * a_s * b_s / (b_s - a_s)
r = np.linspace(a_s, b_s, 20001)
u = 0.5 * EPS0 * (K * Q / r**2) ** 2                                  # energy density between the shells
U_field = np.trapezoid(u * 4 * np.pi * r**2, r)
print(f"spherical capacitor: C = {C_sph * 1e12:.2f} pF,  U = Q^2/2C = {Q**2 / (2 * C_sph):.4e} J,  U = integral of eps0 E^2 / 2 = {U_field:.4e} J")

# Parallel plates on a grid: energy per unit depth, then C = 2U / V0^2, compared with eps0 W / d per unit depth.
n = 81
x = np.linspace(0, 1, n); h = x[1] - x[0]
V0 = np.zeros((n, n)); V0[:, 0] = V0[:, -1] = x; V0[-1, :] = 1.0     # plates at x = 0 and x = 1, W = 1, d = 1; sides hold the linear ramp
Vp = relax.solve_sparse(V0, np.zeros((n, n), dtype=bool))
U_grid = relax.field_energy(Vp, h)
print(f"parallel plates on the grid: U = {U_grid:.4e} J/m,  C = 2U/V0^2 = {2 * U_grid:.4e} F/m,  eps0 W/d = {EPS0:.4e} F/m,  ratio {2 * U_grid / EPS0:.6f}")

### Exercise: a capacitance by two routes

Replace the body of `spherical_capacitor_C` so that it returns the capacitance of two concentric conducting spheres of radii $a < b$ from the closed form of worked example 9. The check computes the same capacitance a second way, without the formula: it integrates the energy density of the Gauss field over the gap for a trial charge $Q$ and uses $C = Q^2/2U$. The two must agree to better than one part in $10^5$ for several radius pairs.

In [ ]:
def spherical_capacitor_C(a, b):
    '''Capacitance of concentric spheres of radii a < b.'''
    return 4 * np.pi * EPS0 * a * b / (b - a)                      # replace with your own expression


Q = 1e-9
for a_, b_ in ((0.1, 0.2), (0.05, 1.0), (1.0, 1.01)):
    r = np.linspace(a_, b_, 400001)
    U = np.trapezoid(0.5 * EPS0 * (K * Q / r**2) ** 2 * 4 * np.pi * r**2, r)
    C_energy = Q**2 / (2 * U)
    print(f"a = {a_}, b = {b_}:  formula {spherical_capacitor_C(a_, b_) * 1e12:10.4f} pF   from field energy {C_energy * 1e12:10.4f} pF")
    assert abs(spherical_capacitor_C(a_, b_) / C_energy - 1) < 1e-5
print("exercise passed")

## 4. Verify

Two closed-form checks of the numerical tools this primer hands to notebooks M2.3 and M2.4.

1. The energy route to capacitance on a staircased geometry. Solve the coaxial capacitor of worked example 8 on a $201\times201$ grid, take $C = 2U/V_0^2$ from `relax.field_energy`, and compare with $2\pi\epsilon_0/\ln(b/a)$. The staircase on the inner circle costs about two percent at this resolution; notebook M2.4 measures how that error falls with refinement.
2. The sphere image of worked example 6, on points that are not on the axis: the potential on the sphere must vanish to rounding error relative to $kq/d$.

In [ ]:
Vc, hc, xs = coax_solution(201, a, b)
C_energy = 2 * relax.field_energy(Vc, hc) / 1.0**2
print(f"coaxial, n = 201: C from energy = {C_energy:.4e} F/m   closed form {C_exact:.4e} F/m   ratio {C_energy / C_exact:.4f}")
assert abs(C_energy / C_exact - 1) < 0.03

rel = np.abs(V_surf).max() / (K * q / d_s)
print(f"sphere image: max |V| on the sphere / (k q / d) = {rel:.1e}")
assert rel < 1e-10
print("verify passed")

## 5. Explore and rebuild

Explore. Put a point charge $q$ at distance $d$ from a grounded sphere of radius $R$ and use the image to compute the induced surface charge, $\sigma = \epsilon_0E_r$ evaluated on the sphere with the field of $q$ and $q'$. Plot $\sigma(\theta)$ for $d = 1.2R$, $2R$ and $5R$, integrate it over the sphere to confirm the total is $-qR/d$, and find the angle beyond which $\sigma$ is negligible. Then compute the force on $q$ from the image and plot it against $d$ on log-log axes: identify the $1/d^3$ law at large $d$ and explain where it comes from (the induced charge shrinks as $1/d$ and its distance from $q$ grows as $d$).

Rebuild. Close this notebook. On paper, write the five properties of a conductor in equilibrium with their one-line reasons, construct the image for a charge above a grounded plane and integrate the induced charge to $-q$, and derive the capacitance of parallel plates, coaxial cylinders and concentric spheres from Gauss' law. Then derive $\tfrac12\epsilon_0E^2$ from the parallel-plate energy. In a blank editor, write the coaxial capacitor on a grid with `relax.solve_sparse`, compute $C$ both from the box flux and from the energy, and reproduce the ratios of section 3.4 and section 4.

## 6. Key results

| result | formula | holds when |
|---|---|---|
| conductor in equilibrium | $\mathbf{E} = 0$, $\rho = 0$ inside; charge on the surface; $V$ constant; $E_n = \sigma/\epsilon_0$ outside | static, ideal conductor |
| shielding | $\mathbf{E} = 0$ in an empty cavity of a conductor | no charge inside the cavity |
| conducting sphere in $E_0$ | $\sigma = 3\epsilon_0E_0\cos\theta$, $p = 4\pi\epsilon_0R^3E_0$ | uniform applied field |
| connected conductors | same $V$; for far-apart spheres $q_1/q_2 = R_1/R_2$, $\sigma\propto1/R$ | equilibrium |
| surface charge from a solver | $\sigma = \epsilon_0E_n$, one-sided difference at the surface | Dirichlet conductor nodes |
| total charge from a solver | $Q = \epsilon_0\oint\mathbf{E}\cdot d\mathbf{S}$ on a box around the conductor | box encloses only that conductor |
| plane image | $-q$ at the mirror point; $\sigma(s) = -\dfrac{qd}{2\pi(s^2 + d^2)^{3/2}}$; total $-q$; $F = \dfrac{q^2}{4\pi\epsilon_0(2d)^2}$ | grounded infinite plane, region above it |
| line charge potential (2D) | $V = -\dfrac{\lambda}{2\pi\epsilon_0}\ln\dfrac{r}{r_0}$; with a plane image $V = \dfrac{\lambda}{2\pi\epsilon_0}\ln\dfrac{r_-}{r_+}$; $\sigma(y) = -\dfrac{\lambda d}{\pi(d^2 + y^2)}$ | uniform along $z$ |
| sphere image | $q' = -qR/d$ at $R^2/d$ from the centre | grounded sphere, $q$ outside |
| capacitance | $C = Q/V$, farads, geometry only | linear (vacuum or linear dielectric) |
| parallel plates | $C = \epsilon_0A/d$ (per unit depth: $\epsilon_0W/d$) | $d\ll$ plate size, fringes ignored |
| coaxial cylinders | $C = 2\pi\epsilon_0/\ln(b/a)$ per unit length | long compared with $b$ |
| concentric spheres | $C = 4\pi\epsilon_0\,ab/(b - a)$; isolated sphere $4\pi\epsilon_0R$ | |
| parallel / series | $C = C_1 + C_2$ (same $V$) ; $1/C = 1/C_1 + 1/C_2$ (same $Q$) | |
| capacitor energy | $U = \tfrac12CV^2 = \tfrac12QV = Q^2/2C$ | |
| energy of charges | $U = \tfrac12\sum_iq_iV_i$ | point charges, $V_i$ from the others |
| field energy | $U = \int\tfrac12\epsilon_0E^2\,dv$ | all space, static |
| capacitance from energy | $C = 2U/V_0^2$ with one conductor at $V_0$, the other at $0$ | the route used in M2.4 |
| floating conductor | its potential minimises the field energy and gives it zero net charge | isolated, initially neutral |

## 7. Warm-up answers

1. If there were a field inside, the mobile charge would move, so the state would not be static. With $\mathbf{E} = 0$ inside, $\nabla\cdot\mathbf{E} = \rho/\epsilon_0$ gives $\rho = 0$: any net charge is on the surface.
2. Mobile charge shifts until its own field cancels the applied one inside: negative charge on the side the field comes from, positive on the far side, zero total. Inside the ball the field is zero. Just outside it is normal to the surface with magnitude $\sigma/\epsilon_0 = 3E_0\cos\theta$, so the field lines bend to meet the ball at right angles and the ball acts as a dipole of moment $4\pi\epsilon_0R^3E_0$.
3. It must satisfy Laplace's equation in the region of interest (the image charges sit outside it) and take the prescribed constant value on the conductor's surface. The uniqueness theorem for the Dirichlet problem then says it is the solution in that region.
4. $C = Q/V$ for a pair of conductors carrying $\pm Q$. No: $V$ is proportional to $Q$ by linearity, so the ratio depends only on the geometry.
5. $\sigma = Q/A$; Gauss gives $E = \sigma/\epsilon_0$ between the plates; $V = Ed = Qd/\epsilon_0A$; so $C = Q/V = \epsilon_0A/d$.
6. $U = \tfrac12CV^2 = \tfrac12QV = Q^2/2C$. Energy density $u = \tfrac12\epsilon_0E^2$, so $U = \int\tfrac12\epsilon_0E^2\,dv$.
7. Series: the same charge $Q$ on every capacitor, voltages add, $1/C = 1/C_1 + 1/C_2$. Parallel: the same voltage, charges add, $C = C_1 + C_2$.